<hr>

<font color=skyblue>**Image Deblurring using Convolutional Neural Networks and Deep Learning**</font>

<hr>

<font color=skyblue>**Part 1: Data preparations**</font>

以下提供兩種模糊影像的處理方式，以做為訓練與驗證資料。分別為：
- Data Processing 1 以小圖做為訓練資料: 將清晰之影像切成小圖（patch），再逐一進行高斯模糊化，並分別儲存清晰與模糊影像。
- Data Processing 2 以大圖做為訓練資料: 直接將原圖進行高斯模糊化，並儲存模糊影像。

<hr>

<font color=skyblue>套件宣告</font>

In [1]:
import os
import cv2
import patchify
import numpy as np
import glob as glob
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from PIL import Image
from tqdm import tqdm

<hr>

<font color=skyblue>**Data Preprocessing 1**: Dataset Patch Creation and blurred patch creation</font>

This script is used to create patches of size (33, 33, 3) from the images in the T91 dataset.
The patches are saved in the following directories:
- inputs/sharp_patches
- inputs/blurred_patches

<hr>

<font color=skyblue>常數宣告</font>

關於製作高斯模糊的指令 cv2.GaussianBlur() 中兩個主要參數：
- kernel size: 決定模糊的程度，數值越大，模糊效果越明顯。通常選擇奇數（如3, 5, 7）以確保中心像素的存在。如圖 1 所示，kernel size 為 (3, 3) 時，模糊效果較輕微；而 kernel size 為 (9, 9) 時，模糊效果較明顯。
- sigmaX: 決定模糊的範圍，數值越大，模糊效果越廣泛。當 kernel size 為 (0, 0) 時，sigmaX 的值會自動計算 kernel size。如圖 2 所示，sigmaX 為 1 時，模糊效果較輕微；而 sigmaX 為 3 時，模糊效果較明顯。

<img src="pictures/Copilot_20260511_122732.png" style="width:350px; height:auto;">
<img src="pictures/Copilot_20260511_123106.png" style="width:350px; height:auto;">




In [2]:
DIR_PATH = 'Deblur/'
SHOW_PATCHES = False # Whether to show the patches. Set to False to speed up the process.
STRIDE = 14 # Stride for the patches.
SIZE = 33 # Size of the patches.
GAUSSIAN_KERNEL_SIZE_PATCH = (0, 0) # Kernel size for Gaussian blur.
SIGMAX = 3 # Sigma for Gaussian blur. 0 means it is calculated from the kernel size.


<font color=skyblue>副程式宣告</font>

In [3]:
def show_patches(patches):
    """
    Function to plot the patches.
    """
    plt.figure(figsize=(patches.shape[0], patches.shape[1]))
    gs = gridspec.GridSpec(patches.shape[0], patches.shape[1])
    gs.update(wspace=0.01, hspace=0.02)
    counter = 0
    for i in range(patches.shape[0]):
        for j in range(patches.shape[1]):
            ax = plt.subplot(gs[counter])
            plt.imshow(patches[i, j, 0, :, :, :])
            plt.axis('off')
            counter += 1
    plt.show()

def create_patches(input_paths, out_sharp_path, out_blur_path):
    '''
    Function to create patches from the images in the input paths.
    The patches are saved in the output paths.'''
    
    os.makedirs(out_sharp_path, exist_ok=True)
    os.makedirs(out_blur_path, exist_ok=True)
    all_paths = []

    for input_path in input_paths:
        all_paths.extend(glob.glob(f"{input_path}/*"))
    print(f"Creating patches for {len(all_paths)} images")

    for image_path in tqdm(all_paths, total=len(all_paths)):
        image = Image.open(image_path)
        image_name = image_path.split(os.path.sep)[-1].split('.')[0]
        
        patches = patchify.patchify(np.array(image), (SIZE, SIZE, 3), STRIDE)
        if SHOW_PATCHES:
            show_patches(patches)

        # print(patches.shape)
        counter = 0
        for i in range(patches.shape[0]):
            for j in range(patches.shape[1]):
                counter += 1
                patch = patches[i, j, 0, :, :, :]
                patch = cv2.cvtColor(patch, cv2.COLOR_RGB2BGR)
                cv2.imwrite(
                    f"{out_sharp_path}/{image_name}_{counter}.png",
                    patch
                )
                blur = cv2.GaussianBlur(patch, GAUSSIAN_KERNEL_SIZE_PATCH, sigmaX=SIGMAX)

                cv2.imwrite(
                    f"{out_blur_path}/{image_name}_{counter}.png",
                    blur
                )


<font color=skyblue>執行圖像裁切</font>

- 清晰小圖另存目錄，作為 labels
- 模糊小圖另存目錄，作為 inputs

In [4]:
src_dir_name = 'T91'
input_paths =  [f"{DIR_PATH}inputs/{src_dir_name}/"]
outputs_sharp_path = DIR_PATH + f"inputs/{src_dir_name}_sharp_patches"
outputs_blur_path = DIR_PATH + f"inputs/{src_dir_name}_blurred_patches_sigmax{SIGMAX}"

create_patches(input_paths, outputs_sharp_path, outputs_blur_path)

Creating patches for 91 images


100%|██████████| 91/91 [00:40<00:00,  2.22it/s]


<hr>

<font color=skyblue>**Data Preprocessing 2**: Prepare blurred images by adding Gaussian blurring to the original (sharp) images</font>

- sharp images : 可以來自 T91, General100, Kaggle_sharp_350
- blurring images : gaussian_blurred，可以使用單一的模糊程度或混合不同模糊程度
- 保留原影像大小

<hr>


<font color=skyblue>常數宣告</font>

In [6]:
DIR_PATH = 'Deblur/'
GAUSSIAN_KERNEL_SIZE_ORIGINAL = (0, 0)
SIGMAX = 3 # Sigma for Gaussian blur. 0 means it is calculated from the kernel size.

<font color=skyblue>執行圖像高斯模糊化</font>

- 保留輸入圖像目錄作為清晰圖（sharp, labels）
- 輸出目錄為模糊圖（blurred）
- 輸出圖像大小與輸入圖像大小相同

In [7]:
src_dir_name = 'General100'
src_dir = DIR_PATH + f'inputs/{src_dir_name}'
images = os.listdir(src_dir)
dst_dir = DIR_PATH + f'inputs/{src_dir_name}_gaussian_blurred_sigmax{SIGMAX}'
os.makedirs(dst_dir, exist_ok=True)

for i, img in tqdm(enumerate(images), total=len(images)):
    img = cv2.imread(f"{src_dir}/{images[i]}", cv2.IMREAD_COLOR)
    # add gaussian blurring
    # blur = cv2.GaussianBlur(img, (5, 5), 0)  # Mild blur
    # blur = cv2.GaussianBlur(img, (15, 15), 5)  # Stronger blur

    blur = cv2.GaussianBlur(img, GAUSSIAN_KERNEL_SIZE_ORIGINAL, sigmaX=SIGMAX)
    cv2.imwrite(f"{dst_dir}/{images[i]}", blur)

print('DONE')

100%|██████████| 100/100 [00:05<00:00, 18.32it/s]

DONE
